In [1]:
# Leading Indicators: backtests
# Zakariya Asif
# Nov 22, 2026.

In [2]:
# Inspecting whether working directory is the project root or ..\src

from pathlib import Path
import pandas as pd

print("Notebook working directory:", Path.cwd())

Notebook working directory: C:\Users\zakar\Jupyter_Projects\4501_leading_indicators_backtests\src


In [3]:
# Working directory is ..\4501_leading_indicators\src
project_root = Path.cwd().parent

data_path = (
    project_root
    / "data"
    / "indicator_screening_2026-11-22.xlsx"
)

if not data_path.is_file():
    raise FileNotFoundError(f"Worbook not found: {data_path}")

In [4]:
# Attempting to load the backtest dataset from the indicator screening matrix workbook
df = pd.read_excel(
    data_path,
    sheet_name="backtest_qrtrly_dataset",
    parse_dates=["Date"],
).dropna(subset=["Date"])

In [5]:
# Validating the imported dataset and giving the columns shorter names for convenience
# Retaining df as the imported source and creating a separate working dataset
import numpy as np

column_names = {
    "Building Permits": "permits",
    "Total Manufacturing Volume": "total_orders",
    "Foreign Manufacturing Volume": "foreign_orders",
    "New Construction Orders Volume": "construction_orders",
    "ifo Business Expectations": "ifo",
    "10-2y Yield Spread": "spread_10_2",
    "10-1y Yield Spread": "spread_10_1",
    "PPP Commerical Electricity Supplied": "electricity_ppi",
    "Real GDP Index": "gdp",
    "Real GDP Index Change (%)": "gdp_qoq_published",
}

data = (
    df.rename(columns=column_names)
    .sort_values("Date")
    .set_index("Date")
    .copy()
)

# Fail visibly if the imported dataset has structural issues
assert data.index.is_unique, "Duplicate dates found."
assert data.index.is_monotonic_increasing, "Dates are not sorted."

expected_quarters = pd.date_range(
    data.index.min(),
    data.index.max(),
    freq="QS-JAN",
)
assert data.index.equals(expected_quarters), (
    "A quarter is missing or a date is not a quarter start."
)

numeric_columns = list(column_names.values())
data[numeric_columns] = data[numeric_columns].apply(
    pd.to_numeric, errors="raise"
)

print("Rows and columns:", data.shape)
print("Coverage:", data.index.min(), "to", data.index.max())

display(data.tail())
display(data[numeric_columns].isna().sum().rename("Missing observations"))

Rows and columns: (87, 11)
Coverage: 2005-01-01 00:00:00 to 2026-07-01 00:00:00


,Quarter,permits,total_orders,foreign_orders,construction_orders,ifo,spread_10_2,spread_10_1,electricity_ppi,gdp,gdp_qoq_published
Date,,,,,,,,,,,
2025-07-01,2025 Q3,22580,85.100000,88.233333,95.000000,90.566667,0.783672,0.820010,122.766667,105.16,0.0
2025-10-01,2025 Q4,21145,93.466667,92.200000,92.300000,90.500000,0.733154,0.787767,123.166667,105.44,0.3
2026-01-01,2026 Q1,22051,89.433333,92.566667,91.366667,88.566667,0.696951,0.766149,114.666667,105.91,0.4
2026-04-01,2026 Q2,21807,90.833333,94.133333,95.700000,84.000000,0.462545,0.556439,113.466667,106.26,0.3
2026-07-01,2026 Q3,20694,91.000000,91.233333,NaN,88.700000,0.405546,0.523954,113.333333,NaN,NaN


permits                0
total_orders           0
foreign_orders         0
construction_orders    1
ifo                    0
spread_10_2            0
spread_10_1            0
electricity_ppi        0
gdp                    1
gdp_qoq_published      1
Name: Missing observations, dtype: int64

In [6]:
# As expected, the missing observations are one each for construction orders, gdp, and gdp & change.
# I'll retain the Q3 row for now and avoid a general data.dropna() at this stage, in order to ascertain which rows are eligible for calculation explicitly.
# Creating the predictor transformations and forecast targets.
# One initial transformation per candidate. The two yield spreads are alternative versions of the same candidate family.
features = pd.DataFrame(index=data.index)

growth_columns = [
    "permits",
    "total_orders",
    "foreign_orders",
    "construction_orders",
    "electricity_ppi",
]

for name in growth_columns:
    features[f"{name}_yoy"] = (
        100 * (data[name] / data[name].shift(4) - 1)
    )

features["ifo"] = data["ifo"]
features["spread_10_2"] = data["spread_10_2"]
features["spread_10_1"] = data["spread_10_1"]

targets = pd.DataFrame(index=data.index)

for horizon in (6, 8):
    targets[f"gdp_growth_{horizon}q"] = (
        100 * (data["gdp"].shift(-horizon) / data["gdp"] - 1)
    )

# GDP-only predictor for the benchmark model
gdp_momentum = 100 * (
    data["gdp"] / data["gdp"].shift(4) - 1
)

display(features.tail())
display(targets.tail(12))

,permits_yoy,total_orders_yoy,foreign_orders_yoy,construction_orders_yoy,electricity_ppi_yoy,ifo,spread_10_2,spread_10_1
Date,,,,,,,,
2025-07-01,16.903961,-2.408257,-1.378539,9.447005,-5.077320,90.566667,0.783672,0.820010
2025-10-01,10.666248,7.145587,3.363229,3.320896,-4.571281,90.500000,0.733154,0.787767
2026-01-01,12.156045,3.311513,3.464978,-1.331893,-8.291122,88.566667,0.696951,0.766149
2026-04-01,9.875548,4.646697,3.747245,5.049396,-7.650570,84.000000,0.462545,0.556439
2026-07-01,-8.352524,6.933020,3.400076,NaN,-7.683953,88.700000,0.405546,0.523954


,gdp_growth_6q,gdp_growth_8q
Date,,
2023-10-01,0.506402,0.745270
2024-01-01,0.266972,0.982075
2024-04-01,0.668322,1.451213
2024-07-01,1.010968,NaN
2024-10-01,1.122954,NaN
2025-01-01,NaN,NaN
2025-04-01,NaN,NaN
2025-07-01,NaN,NaN
2025-10-01,NaN,NaN


In [7]:
# The 6 and 8 quarter growth rates for GDP correspond to the distance between forecast targets of 2027 Q4 and 2028 Q2 - compared to the reference quarter for which data is available - i.e., 2026 Q2
# The blank forward targets near the end of the dataset are expected as those future GDP observations have not yet occured (over the 6-8 quarter period).
# Defining the forecast and target-availability dates.
# For the first comparison, I'll retain the previously documented convention of forecasting 65 calendar days after the reference quarter ends, with the detailed GDP outcome treated as available 55 days after its quarter ends.
reference_quarters = data.index.to_period("Q")

forecast_dates = pd.Series(
    reference_quarters.to_timestamp(how="end").normalize()
    + pd.Timedelta(days=65),
    index=data.index,
    name="forecast_date",
)

target_dates = pd.DataFrame(index=data.index)
target_available = pd.DataFrame(index=data.index)

for horizon in (6, 8):
    ending_quarters = reference_quarters + horizon

    target_dates[f"{horizon}q"] = (
        ending_quarters.to_timestamp(how="start")
    )

    target_available[f"{horizon}q"] = (
        ending_quarters.to_timestamp(how="end").normalize()
        + pd.Timedelta(days=55)
    )

reference_quarter = pd.Timestamp("2026-04-01")

display(pd.DataFrame({
    "Target quarter": [
        target_dates.loc[reference_quarter, "6q"],
        target_dates.loc[reference_quarter, "8q"],
    ],
    "Horizon": [6, 8],
}))

,Target quarter,Horizon
0,2027-10-01,6
1,2028-04-01,8


In [8]:
# Running the first historical forecast as an example in order to understand the back-test convention from my methodological sources.
## Reference quarter: 2015 Q1
## Horizon: eight quarters
## Predictor: the four-quarter change in building permits
## Outcome: cumulative real GDP growth from 2015 Q1 to 2017 Q1

example_origin = pd.Timestamp("2015-01-01") # 2015 Q1
example_horizon = 8
example_predictor = "permits_yoy"

example_forecast_date = forecast_dates.loc[example_origin]
example_target_quarter = target_dates.loc[
    example_origin, f"{example_horizon}q"
    ]

print("Reference quarter:", example_origin.date())
print("Assumed forecast date:", example_forecast_date.date())
print("Target quarter:", example_target_quarter.date())

Reference quarter: 2015-01-01
Assumed forecast date: 2015-06-04
Target quarter: 2017-01-01


In [9]:
# Combining candidate indicators with the GDP benchmark predictor
backtest_inputs = features.copy()
backtest_inputs["gdp_momentum"] = gdp_momentum

# A row is complete only if every candidate and GDP momentum is available
common_complete = backtest_inputs.notna().all(axis=1)

print("Complete rows:", common_complete.sum())
print("Incomplete rows:", (~common_complete).sum())

display(backtest_inputs.head(6))

Complete rows: 82
Incomplete rows: 5


,permits_yoy,total_orders_yoy,foreign_orders_yoy,construction_orders_yoy,electricity_ppi_yoy,ifo,spread_10_2,spread_10_1,gdp_momentum
Date,,,,,,,,,
2005-01-01,NaN,NaN,NaN,NaN,NaN,95.966667,1.158595,1.394667,NaN
2005-04-01,NaN,NaN,NaN,NaN,NaN,93.200000,1.129401,1.271169,NaN
2005-07-01,NaN,NaN,NaN,NaN,NaN,97.366667,0.954139,1.069454,NaN
2005-10-01,NaN,NaN,NaN,NaN,NaN,99.900000,0.721616,0.891847,NaN
2006-01-01,30.536147,10.200364,13.712544,6.520657,4.317789,101.966667,0.521516,0.706976,2.899251
2006-04-01,8.475861,10.592460,13.463463,8.637874,4.994325,101.333333,0.604680,0.834074,4.072072


In [10]:
# Creating a filter for the training data available at the example forecast date, in order to identify which historical outcomes were available
# The desired predictive outcome at every historical reference quarter
example_target = targets[f"gdp_growth_{example_horizon}q"]

# When each historical outcome is assumed to have become available
example_target_available = target_available[f"{example_horizon}q"]

# Selecting eligible training rows
training_filter = (
    common_complete
    & example_target.notna()
    & (data.index < example_origin)
    & (example_target_available <= example_forecast_date)
)

training_dates = data.index[training_filter]

print("Number of training observations:", len(training_dates))
print("First training reference quarter:", training_dates.min().date())
print("Last training reference quarter:", training_dates.max().date())
print(
    "Latest traning outcome available:",
    example_target_available.loc[training_dates].max().date()
)

Number of training observations: 29
First training reference quarter: 2006-01-01
Last training reference quarter: 2013-01-01
Latest traning outcome available: 2015-05-25


In [11]:
# Condition and purpose:
## common_complete - All candidate inputs are present for that historical row.
## example_target.notna() - The eventual GDP outcome exists in the dataset.
## data.index < example_origin - The training reference quarter precedes the quarter being forecast.
## example_target_available <= example_forecast_date - The outcome had become available by the assumed forecast date.

In [12]:
# Inspecting training rows to inspect what will enter the regression model
training_example = pd.DataFrame({
    "Building permits YoY (%)":
        features.loc[training_dates, example_predictor],
    "Future GDP growth (%)":
        example_target.loc[training_dates],
    "GDP target quarter":
        target_dates.loc[training_dates, f"{example_horizon}q"],
    "GDP outcome available":
        example_target_available.loc[training_dates],
})

display(training_example.head())
display(training_example.tail())

,Building permits YoY (%),Future GDP growth (%),GDP target quarter,GDP outcome available
Date,,,,
2006-01-01,30.536147,6.750411,2008-01-01,2008-05-25
2006-04-01,8.475861,4.547553,2008-04-01,2008-08-24
2006-07-01,-3.549116,3.195876,2008-07-01,2008-11-24
2006-10-01,-27.651649,0.146760,2008-10-01,2009-02-24
2007-01-01,-51.251628,-4.679220,2009-01-01,2009-05-25


,Building permits YoY (%),Future GDP growth (%),GDP target quarter,GDP outcome available
Date,,,,
2012-01-01,-9.076956,2.421883,2014-01-01,2014-05-25
2012-04-01,2.527737,2.342906,2014-04-01,2014-08-24
2012-07-01,4.357104,2.673854,2014-07-01,2014-11-24
2012-10-01,-14.468408,3.827441,2014-10-01,2015-02-24
2013-01-01,4.861580,4.163043,2015-01-01,2015-05-25


In [13]:
# Predictor values for the eligible training quarters
x_train = features.loc[
    training_dates, example_predictor
    ].to_numpy()

# Corresponding future GDP growth outcomes
y_train = example_target.loc[
    training_dates
    ].to_numpy()

print("Predictor observations:", len(x_train))
print("GDP outcome observations:", len(y_train))

print("\nFirst five predictor values:", x_train[:5])
print("First five GDP outcomes:", y_train[:5])

Predictor observations: 29
GDP outcome observations: 29

First five predictor values: [ 30.53614682   8.47586085  -3.54911637 -27.65164874 -51.2516281 ]
First five GDP outcomes: [ 6.75041089  4.54755309  3.19587629  0.14675999 -4.67921975]


In [14]:
# Each permits value is paired with the GDP growth outcome for the same historical reference quarter.
# Fitting the OLS regression
# Adding a column of ones so the regression estimates an intecept
X_train = np.column_stack([
    np.ones(len(x_train)),
    x_train
])

# Fit ordinary least squares
coefficients, residuals, rank, singular_values = np.linalg.lstsq(
    X_train,
    y_train,
    rcond=None
)

intercept = coefficients[0]
permits_coefficient = coefficients[1]

print(f"Intercept: {intercept:.4f}")
print(f"Permits coefficient: {permits_coefficient:.4f}")
print(f"Number of training observations: {len(y_train)}")
print(f"Design matrix rank: {rank}")

Intercept: 2.1559
Permits coefficient: 0.1159
Number of training observations: 29
Design matrix rank: 2


In [15]:
# The fitted equation above is:
## GDP growth over 8 quarters = 2.1559 + 0.1159(permits YoY)
# OLS chooses the intercept and coefficient which minimizes the sum of squared errors across the 29 training observations.
# A matrix rank of 2 indicates that, for this simple mode, both the intercept and slope can be estimated separately. i.e., two linearly independent rows/columns.
# The coefficient suggests that, within this fitted historical relationship, a one-percentage-point increase in permits YoY is assocation with a ~0.116 percentage point increase in cumulative GDP growth over the subsequent 8 quarters. It does not necessarily suggest that permits result in (i.e., cause) GDP growth.

In [16]:
# Building the historical forecast
# Permits reading at the 2015 Q1 reference quarter
example_x = features.loc[
    example_origin, example_predictor
    ]

# Applying the fitted equation
example_prediction = (
    intercept + permits_coefficient * example_x
)

print(f"2015 Q1 permits YoY: {example_x:.4f}%")
print(f"Predicted 8-quarter GDP growth: {example_prediction:.4f}%")

2015 Q1 permits YoY: -1.9178%
Predicted 8-quarter GDP growth: 1.9336%


In [17]:
# The model above therefore predicts that the GDP index would increase by ~1.93% in total from 2015 Q1 to 2017 Q1. 
# This is cumulative growth over two years, not an annual growth rate.

In [18]:
# Comparing the forecast with the outcome
# The actual outcome is used only after the forecast has been made
example_actual = example_target.loc[example_origin]

# Using the established error convention
example_error = example_prediction - example_actual

print(f"Forecast: {example_prediction:.4f}%")
print(f"Actual GDP growth: {example_actual:.4f}%")
print(f"Forecast error: {example_error:.4f} percentage points")

Forecast: 1.9336%
Actual GDP growth: 5.0402%
Forecast error: -3.1066 percentage points


In [19]:
# Attempting to start building the actual forecast loop now.
# Setting up two reference quarters for the first loop
test_origins = pd.to_datetime([
    "2015-01-01", # 2015 Q1
    "2015-04-01", # 2015 Q2
])

# Store one result per forecast
test_results = []

# test_origins contains the quarters I want to forecast from. test_results starts as an empty list; the loop will add one record for each forecast.

In [20]:
# Building and running the two-quarter loop
for origin in test_origins:
    # 1. Identifying the assumed date of this forecast
    forecast_date = forecast_dates.loc[origin]

    # 2. Selecting the GDP target and its assumed availability dates
    target = targets[f"gdp_growth_{example_horizon}q"]

    outcome_available = target_available[
        f"{example_horizon}q"
        ]

    # 3. Identifying eligible training observations
    training_filter = (
    common_complete
    & target.notna()
    & (data.index < origin)
    & (outcome_available <= forecast_date)
    )

    training_dates = data.index[training_filter]

    # Stop visibly if something has gone wrong
    if len(training_dates) < 28:
        raise ValueError(
            f"Only {len(training_dates)} training observations "
            f"for {origin.date()}."
        )

    # 4. Preparing the training predictor and GDP outcomes
    x_train = features.loc[
        training_dates, example_predictor
        ].to_numpy()

    y_train = target.loc[
        training_dates
        ].to_numpy()

    X_train = np.column_stack([
        np.ones(len(x_train)),
        x_train
    ])

    # 5. Fitting the OLS regression
    coefficients, residuals, rank, singular_values = np.linalg.lstsq(
        X_train,
        y_train,
        rcond=None
    )

    if rank !=2:
        raise ValueError(
            f"Regression cannot estimate both coefficients "
            f"for {origin.date()}."
        )

    intercept = coefficients[0]
    permits_coefficient = coefficients[1]

    # 6. Calculate the historical forecast
    predictor_value = features.loc[
        origin, example_predictor
        ]

    prediction = (
        intercept
        + permits_coefficient * predictor_value
    )

    # 7. Compare with the eventual outcome
    actual = target.loc[origin]
    error = prediction - actual

    # 8. Save the inputs, coefficients, forecast and outcome
    test_results.append({
        "Reference quarter": origin,
        "Forecast date": forecast_date,
        "Target quarter": target_dates.loc[
            origin, f"{example_horizon}q"
            ],
        "Training observations": len(training_dates),
        "First training quarter": training_dates.min(),
        "Last training quarter": training_dates.max(),
        "Latest training outcome available":
            outcome_available.loc[training_dates].max(),
        "Intercept": intercept,
        "Permits coefficient": permits_coefficient,
        "Permits YoY": predictor_value,
        "Forecast": prediction,
        "Actual": actual,
        "Error": error,
    })

    # Turn the saved records into a table
test_results_df = pd.DataFrame(test_results)

display(test_results_df)

,Reference quarter,Forecast date,Target quarter,Training observations,First training quarter,Last training quarter,Latest training outcome available,Intercept,Permits coefficient,Permits YoY,Forecast,Actual,Error
0,2015-01-01,2015-06-04,2017-01-01,29,2006-01-01,2013-01-01,2015-05-25,2.155890,0.115932,-1.917834,1.933551,5.040175,-3.106624
1,2015-04-01,2015-09-03,2017-04-01,30,2006-01-01,2013-04-01,2015-08-24,2.211055,0.115542,0.887189,2.313562,5.174738,-2.861176


In [21]:
# Checking the first iteration against the existing example
display(
    test_results_df[[
        "Reference quarter",
        "Training observations",
        "Intercept",
        "Permits coefficient",
        "Permits YoY",
        "Forecast",
        "Actual",
        "Error",
    ]].round(4)
)

,Reference quarter,Training observations,Intercept,Permits coefficient,Permits YoY,Forecast,Actual,Error
0,2015-01-01,29,2.1559,0.1159,-1.9178,1.9336,5.0402,-3.1066
1,2015-04-01,30,2.2111,0.1155,0.8872,2.3136,5.1747,-2.8612


In [22]:
# Validating training cutoffs
display(
    test_results_df[[
        "Reference quarter",
        "Forecast date",
        "Last training quarter",
        "Latest training outcome available",
        "Target quarter",
    ]]
)

assert (
    test_results_df["Latest training outcome available"]
    <= test_results_df["Forecast date"]
).all()

print("All saved training outcomes were available by the assumed forecast dates.")

,Reference quarter,Forecast date,Last training quarter,Latest training outcome available,Target quarter
0,2015-01-01,2015-06-04,2013-01-01,2015-05-25,2017-01-01
1,2015-04-01,2015-09-03,2013-04-01,2015-08-24,2017-04-01


All saved training outcomes were available by the assumed forecast dates.


In [23]:
# Choosing the historical range
# Initial model development period:
## First reference quarter: 2015 Q1
## Last reference quarter: 2018 Q3
## Horizon: eight quarters
## Predictor: building permits YoY

# First development-period back-test
development_start = pd.Timestamp("2015-01-01") # 2015 Q1
development_end = pd.Timestamp("2018-07-01") # 2018 Q3

# The hypothetical date by which development outcomes must be available/matured. 
# Ensures that an outcome is not used to select a model before the outcome was available under timing assumptions
selection_date = pd.Timestamp("2020-12-31")

horizon = 8
predictor = "permits_yoy"
minimum_training_observations = 28

# Selecting the reference quarters from the existing quarterly index
development_origins = data.index[
    (data.index >= development_start)
    & (data.index <= development_end)
    ]

print("Number of reference quarters:", len(development_origins))
print("First:", development_origins.min().date())
print("Last:", development_origins.max().date())

Number of reference quarters: 15
First: 2015-01-01
Last: 2018-07-01


In [24]:
# Running the expanding-window back-test
# Follows the same steps as the two-quarter loop, with a few additional checks

# Select the eight-quarter GDP target and availability dates
target = targets[f"gdp_growth_{horizon}q"]

outcome_available = target_available[f"{horizon}q"]

# Start a fresh list each time this cell runs
development_results = []

for origin in development_origins:

    # Forecast date and eventual target quarter
    forecast_date = forecast_dates.loc[origin]

    target_quarter = target_dates.loc[
        origin, f"{horizon}q"
        ]

    # Verify this forecast can be evaluated for model selection
    if pd.isna(target.loc[origin]):
        raise ValueError(
            f"GDP outcome is missing for {origin.date()}."
        )

    if outcome_available.loc[origin] > selection_date:
        raise ValueError(
            f"The outcome for {origin.date()} was not available "
            f" by the selection date."
        )

    if not common_complete.loc[origin]:
        raise ValueError(
            f"Common predictor inputs are incomplete "
            f"for {origin.date()}."
        )

    # Select training outcomes available at the forecast date
    training_filter = (
        common_complete
        & target.notna()
        & (data.index < origin)
        & (outcome_available <= forecast_date)
    )

    training_dates = data.index[training_filter]

    if len(training_dates) < minimum_training_observations:
        raise ValueError(
            f"Only {len(training_dates)} training observations "
            f"for {origin.date()}."
        )

    # Preparing the training data
    x_train = features.loc[
        training_dates, predictor
    ].to_numpy()

    y_train = target.loc[
        training_dates
    ].to_numpy()

    X_train = np.column_stack([
        np.ones(len(x_train)),
        x_train,
    ])

    # Fitting the regression for this forecast date
    coefficients, residuals, rank, singular_values = np.linalg.lstsq(
        X_train,
        y_train,
        rcond=None
    )

    if rank !=2:
        raise ValueError(
            f"Regression cannot estimate both coefficients "
            f"for {origin.date()}."
        )

    intercept = coefficients[0]
    permits_coefficient = coefficients[1]

    # Producing and evaluating the forecast
    predictor_value = features.loc[origin, predictor]

    prediction = (
        intercept + permits_coefficient * predictor_value)

    actual = target.loc[origin]
    error = prediction - actual

    # Saving the information needed to inspect the forecast
    development_results.append({
        "Reference quarter": origin,
        "Forecast date": forecast_date,
        "Target quarter": target_quarter,
        "Target available": outcome_available.loc[origin],
        "Training observations": len(training_dates),
        "First training quarter": training_dates.min(),
        "Last training quarter": training_dates.max(),
        "Latest training outcome available":
            outcome_available.loc[training_dates].max(),
        "Intercept": intercept,
        "Permits coefficient": permits_coefficient,
        "Permits YoY": predictor_value,
        "Forecast": prediction,
        "Actual": actual,
        "Error": error,
    })

    # (outside the intended loop) Build and show the final table once
development_results_df = pd.DataFrame(development_results)

display(development_results_df)

,Reference quarter,Forecast date,Target quarter,Target available,Training observations,First training quarter,Last training quarter,Latest training outcome available,Intercept,Permits coefficient,Permits YoY,Forecast,Actual,Error
0,2015-01-01,2015-06-04,2017-01-01,2017-05-25,29,2006-01-01,2013-01-01,2015-05-25,2.155890,0.115932,-1.917834,1.933551,5.040175,-3.106624
1,2015-04-01,2015-09-03,2017-04-01,2017-08-24,30,2006-01-01,2013-04-01,2015-08-24,2.211055,0.115542,0.887189,2.313562,5.174738,-2.861176
2,2015-07-01,2015-12-04,2017-07-01,2017-11-24,31,2006-01-01,2013-07-01,2015-11-24,2.248587,0.115967,11.038426,3.528685,5.541224,-2.012539
3,2015-10-01,2016-03-05,2017-10-01,2018-02-24,32,2006-01-01,2013-10-01,2016-02-24,2.289724,0.116684,12.843981,3.788410,6.087046,-2.298635
4,2016-01-01,2016-06-04,2018-01-01,2018-05-25,33,2006-01-01,2014-01-01,2016-05-25,2.323935,0.117180,20.588517,4.736504,4.650926,0.085578
5,2016-04-01,2016-09-03,2018-04-01,2018-08-24,34,2006-01-01,2014-04-01,2016-08-24,2.382242,0.116586,8.713191,3.398078,5.063933,-1.665855
6,2016-07-01,2016-12-04,2018-07-01,2018-11-24,35,2006-01-01,2014-07-01,2016-11-24,2.448731,0.114580,-5.527501,1.815391,4.184354,-2.368963
7,2016-10-01,2017-03-06,2018-10-01,2019-02-24,36,2006-01-01,2014-10-01,2017-02-24,2.472795,0.114864,1.320704,2.624496,4.115102,-1.490606
8,2017-01-01,2017-06-04,2019-01-01,2019-05-25,37,2006-01-01,2015-01-01,2017-05-25,2.547812,0.114520,-11.853854,1.190303,3.536658,-2.346355
9,2017-04-01,2017-09-03,2019-04-01,2019-08-24,38,2006-01-01,2015-04-01,2017-08-24,2.614784,0.115094,-4.222635,2.128785,2.760797,-0.632012


In [25]:
# Changes from the test loop for above:
## The loop runs over development_origins instead of two manually selected dates.
## development_results = [] sits at the start of the same cell, so rerunning the cell produces a fresh result rather than appending duplicates.
## It checks that each forecast's eventual outcome was available by the selection date.
## It checks that the forecast-quarter inputs are complete. 
## It constructs and displays the results table after the loop has finished.
# The training sample expands as successive GDP outcomes mature. The OLS model is fitted again at each origin.

In [26]:
# Validating the full set of forecasts

# Checking expected number of rows
assert len(development_results_df) == len(development_origins)

# Checking that each origin appears only once
assert development_results_df["Reference quarter"].is_unique

# Checking that training outcomes were available by each forecast date
assert (
    development_results_df["Latest training outcome available"]
    <= development_results_df["Forecast date"]
).all()

# Checking that every evaluation outcome was available by selection
assert (
    development_results_df["Target available"]
    <= selection_date
).all()

# Checking that saved errors equal forecast minus actual
assert np.allclose(
    development_results_df["Error"],
    development_results_df["Forecast"] - development_results_df["Actual"]
)

print("All development-period checks passed.")

display(
    development_results_df[[
        "Reference quarter",
        "Forecast date",
        "Target quarter",
        "Training observations",
        "Forecast",
        "Actual",
        "Error",
    ]].round(4)
)

# I expect to see 15 rows, with the first two reproducing the forecasts from the test loop
# The checks validate the timing logic under the assumed release delay
# However, they do not prove that those delays match every historical release or that current-vintage values were known at the time.

All development-period checks passed.


,Reference quarter,Forecast date,Target quarter,Training observations,Forecast,Actual,Error
0,2015-01-01,2015-06-04,2017-01-01,29,1.9336,5.0402,-3.1066
1,2015-04-01,2015-09-03,2017-04-01,30,2.3136,5.1747,-2.8612
2,2015-07-01,2015-12-04,2017-07-01,31,3.5287,5.5412,-2.0125
3,2015-10-01,2016-03-05,2017-10-01,32,3.7884,6.0870,-2.2986
4,2016-01-01,2016-06-04,2018-01-01,33,4.7365,4.6509,0.0856
5,2016-04-01,2016-09-03,2018-04-01,34,3.3981,5.0639,-1.6659
6,2016-07-01,2016-12-04,2018-07-01,35,1.8154,4.1844,-2.3690
7,2016-10-01,2017-03-06,2018-10-01,36,2.6245,4.1151,-1.4906
8,2017-01-01,2017-06-04,2019-01-01,37,1.1903,3.5367,-2.3464
9,2017-04-01,2017-09-03,2019-04-01,38,2.1288,2.7608,-0.6320


In [27]:
# Calculating MAE, RMSE, and bias
errors = development_results_df["Error"]

# Mean absolute error
mae = errors.abs().mean()

# Root mean squared error
rmse = np.sqrt((errors ** 2).mean())

# Average signed forecast error
bias = errors.mean()

development_summary = pd.DataFrame([{
    "Predictor": predictor,
    "Horizon (quarters)": horizon,
    "First reference quarter":
        development_results_df["Reference quarter"].min(),
    "Last reference quarter":
        development_results_df["Reference quarter"].max(),
    "Forecasts": len(development_results_df),
    "MAE": mae,
    "RMSE": rmse,
    "Bias": bias,
}])

display(development_summary.round(4))

,Predictor,Horizon (quarters),First reference quarter,Last reference quarter,Forecasts,MAE,RMSE,Bias
0,permits_yoy,8,2015-01-01,2018-07-01,15,2.693,3.9469,0.1887


In [28]:
# All three measures are in percentage points of cumulative eight-quarter GDP growth.
## MAE = 2.693: the forecasts missed the actual outcome by an average of about 2.693 percentage points, ignoring direction.
## RMSE = 3.9469: larger errors receive more weight; its weight being larger than MAE indicates that some forecasts had relatively large errors.
## Bias = + 0.1887: the signed errors average slightly positive, indicating slight average overprediction under the forecast - actual convention. Positive and negative errors can cancel out, so the small bias does not necessarily indicate the forecasts were consistently close.
# These numbers describe the permits model over this development period. In order to select a good model, I'll need to compare it against benchmarks and other candidates over the same forecast origins.

In [29]:
# Exporting the results to designated destination folder within the Jupyter project
output_dir = project_root / "backtest_first_run_results"

# Detailed forecast-by-forecast results
detail_path = output_dir / "permits_8q_development_detail.csv"

# Summary error measures
summary_path = output_dir / "permits_8q_development_summary.csv"

development_results_df.to_csv(
    detail_path,
    index=False,
    date_format="%Y-%m-%d"
)

development_summary.to_csv(
    summary_path,
    index=False,
    date_format="%Y-%m-%d"
)

print("Saved:", detail_path)
print("Saved:", summary_path)

Saved: C:\Users\zakar\Jupyter_Projects\4501_leading_indicators_backtests\backtest_first_run_results\permits_8q_development_detail.csv
Saved: C:\Users\zakar\Jupyter_Projects\4501_leading_indicators_backtests\backtest_first_run_results\permits_8q_development_summary.csv


In [30]:
# Adding the historical-mean benchmark
# It will use the same training quarters as the permits model at each origin
# Its forecast will change as new outcomes become available

# Historical-mean benchmark for the same development origins
benchmark_results = []

for origin in development_origins:

    forecast_date = forecast_dates.loc[origin]

    # Using the same eligibility rules as the permits model
    training_filter = (
        common_complete
        & target.notna()
        & (data.index < origin)
        & (outcome_available <= forecast_date)
    )

    training_dates = data.index[training_filter]

    if len(training_dates) < minimum_training_observations:
        raise ValueError(
            f"Only {len(training_dates)} training observations "
            f"for {origin.date()}."
        )

    # Forecast = ean of the eliginle historical GDP outcomes
    prediction = target.loc[training_dates].mean()

    actual = target.loc[origin]
    error = prediction - actual

    benchmark_results.append({
        "Reference quarter": origin,
        "Forecast date": forecast_date,
        "Target quarter": target_dates.loc[
            origin, f"{horizon}q"
        ],
        "Training observations": len(training_dates),
        "Latest training outcome available":
            outcome_available.loc[training_dates].max(),
        "Forecast": prediction,
        "Actual": actual,
        "Error": error,
    })

benchmark_results_df = pd.DataFrame(benchmark_results)

display(
    benchmark_results_df[[
        "Reference quarter",
        "Training observations",
        "Forecast",
        "Actual",
        "Error",
    ]].round(4)
)

# The benchmark does not use the current permits reading. It predicts future GDP growth from the mean of past completed GDP-growth outcomes.

,Reference quarter,Training observations,Forecast,Actual,Error
0,2015-01-01,29,2.0385,5.0402,-3.0017
1,2015-04-01,30,2.0868,5.1747,-3.0880
2,2015-07-01,31,2.1350,5.5412,-3.4063
3,2015-10-01,32,2.1916,6.0870,-3.8954
4,2016-01-01,33,2.2381,4.6509,-2.4128
5,2016-04-01,34,2.2887,5.0639,-2.7753
6,2016-07-01,35,2.3346,4.1844,-1.8498
7,2016-10-01,36,2.3669,4.1151,-1.7482
8,2017-01-01,37,2.4392,3.5367,-1.0975
9,2017-04-01,38,2.5112,2.7608,-0.2496


In [31]:
# Comparing the benchmark with permits

# Verifying that both models were evaluated on the same quarters
assert(
    benchmark_results_df["Reference quarter"].equals(
        development_results_df["Reference quarter"]
    )
)

assert np.array_equal(
    benchmark_results_df["Training observations"],
    development_results_df["Training observations"]
)

assert np.allclose(
    benchmark_results_df["Actual"],
    development_results_df["Actual"]
)

# Calculating the benchmark's error measures
benchmark_errors = benchmark_results_df["Error"]

benchmark_summary = pd.DataFrame([{
    "Model": "Historical mean",
    "Forecasts": len(benchmark_errors),
    "MAE": benchmark_errors.abs().mean(),
    "RMSE": np.sqrt((benchmark_errors ** 2).mean()),
    "Bias": benchmark_errors.mean(),
}])

# Putting the permits results into the same format
permits_summary = pd.DataFrame([{
    "Model": "Building permits YoY",
    "Forecasts": len(development_results_df),
    "MAE": development_results_df["Error"].abs().mean(),
    "RMSE": np.sqrt(
        (development_results_df["Error"] ** 2).mean()
    ),
    "Bias": development_results_df["Error"].mean(),
}])

benchmark_comparison = pd.concat(
    [benchmark_summary, permits_summary],
    ignore_index=True
)

display(benchmark_comparison.round(4))

,Model,Forecasts,MAE,RMSE,Bias
0,Historical mean,15,3.0565,4.1862,-0.0801
1,Building permits YoY,15,2.6930,3.9469,0.1887


In [32]:
# Interpretation: permits has lower MAE and RMSE in this development sample, so it improves on the historical mean under both measures.
# The margin is modest and the 15 overlapping forecasts do not establish that the difference will persist.

In [33]:
# Expanding the loop to each candidate indicator using the same forecast origins, target, and training-date rules for eight variants
# This is still one predictor per regression; will test pairs in a later step
# Inspecting the predictor names first
candidate_predictors = features.columns.tolist()

print("Number of predictor variants:", len(candidate_predictors))
print(candidate_predictors)

Number of predictor variants: 8
['permits_yoy', 'total_orders_yoy', 'foreign_orders_yoy', 'construction_orders_yoy', 'electricity_ppi_yoy', 'ifo', 'spread_10_2', 'spread_10_1']


In [34]:
# One regression per indicator, at each development origin
single_indicator_results = []

for origin in development_origins:

    forecast_date = forecast_dates.loc[origin]

    # Using the same training dates for each candidate - as established above
    training_filter = (
        common_complete
        & target.notna()
        & (data.index < origin)
        & (outcome_available <= forecast_date)
    )

    training_dates = data.index[training_filter]

    if len(training_dates) < minimum_training_observations:
        raise ValueError(
            f"Only {len(training_dates)} training observations "
            f"for {origin.date()}."
        )

    if not common_complete.loc[origin]:
        raise ValueError(
            f"Candidate inputs are incomplete for {origin.date()}."
        )

    if pd.isna(target.loc[origin]):
        raise ValueError(
            f"GDP outcome is missing for {origin.date()}."
        )

    if outcome_available.loc[origin] > selection_date:
        raise ValueError(
            f"Outcome for {origin.date()} was unavailable "
            f"by the selection date."
        )

    y_train = target.loc[training_dates].to_numpy()

    for candidate in candidate_predictors:

        # Training values for candidate
        x_train = features.loc[
            training_dates, candidate
            ].to_numpy()

        X_train = np.column_stack([
            np.ones(len(x_train)),
            x_train,
        ])

        # Fitting a separate OLS model for candidate and origin
        coefficients, residuals, rank, singular_values = np.linalg.lstsq(
            X_train,
            y_train,
            rcond=None
        )

        if rank != 2:
            raise ValueError(
                f"Cannot estimate {candidate} for {origin.date()}."
            )

        intercept = coefficients[0]
        coefficient = coefficients[1]

        predictor_value = features.loc[origin, candidate]

        prediction = (
            intercept + coefficient * predictor_value
        )

        actual = target.loc[origin]
        error = prediction - actual

        single_indicator_results.append({
            "Predictor": candidate,
            "Reference quarter": origin,
            "Forecast date": forecast_date,
            "Target quarter": target_dates.loc[
                origin, f"{horizon}q"
            ],
            "Target available": outcome_available.loc[origin],
            "Training observations": len(training_dates),
            "First training quarter": training_dates.min(),
            "Last training quarter": training_dates.max(),
            "Latest training outcome available":
                outcome_available.loc[training_dates].max(),
            "Intercept": intercept,
            "Coefficient": coefficient,
            "Predictor value": predictor_value,
            "Forecast": prediction,
            "Actual": actual,
            "Error": error,
        })

single_indicator_results_df = pd.DataFrame(
    single_indicator_results
)

print("Forecast rows:", len(single_indicator_results_df))
display(single_indicator_results_df.head())

Forecast rows: 120


,Predictor,Reference quarter,Forecast date,Target quarter,Target available,Training observations,First training quarter,Last training quarter,Latest training outcome available,Intercept,Coefficient,Predictor value,Forecast,Actual,Error
0,permits_yoy,2015-01-01,2015-06-04,2017-01-01,2017-05-25,29,2006-01-01,2013-01-01,2015-05-25,2.155890,0.115932,-1.917834,1.933551,5.040175,-3.106624
1,total_orders_yoy,2015-01-01,2015-06-04,2017-01-01,2017-05-25,29,2006-01-01,2013-01-01,2015-05-25,2.241609,-0.084401,0.603774,2.190650,5.040175,-2.849526
2,foreign_orders_yoy,2015-01-01,2015-06-04,2017-01-01,2017-05-25,29,2006-01-01,2013-01-01,2015-05-25,2.274997,-0.068116,0.392619,2.248253,5.040175,-2.791922
3,construction_orders_yoy,2015-01-01,2015-06-04,2017-01-01,2017-05-25,29,2006-01-01,2013-01-01,2015-05-25,2.190639,-0.163873,-1.013942,2.356797,5.040175,-2.683379
4,electricity_ppi_yoy,2015-01-01,2015-06-04,2017-01-01,2017-05-25,29,2006-01-01,2013-01-01,2015-05-25,-0.207840,0.481994,-0.223881,-0.315749,5.040175,-5.355925


In [35]:
# 120 rows:
## 15 reference quarters x 8 predictor variants = 120 forecasts
# There are now two loops:
## The outer loop moves through reference quarters
## The inner loop fits each candidate model for that quarter
# All eight models use the same training dates at a given origin, making their error measures comparable.

In [36]:
# Validating the expanded loop

# Each predictor should have one forecast per development quarter
counts = single_indicator_results_df.groupby(
    "Predictor"
)["Reference quarter"].nunique()

assert (counts == len(development_origins)).all()

# No predictor-origin combination should occur twice
assert not single_indicator_results_df.duplicated(
    ["Predictor", "Reference quarter"]
).any()

# Training outcomes must be available by the forecast date
assert (
    single_indicator_results_df["Latest training outcome available"]
    <= single_indicator_results_df["Forecast date"]
).all()

# Evaluation outcomes must be available by the selection date
assert (
    single_indicator_results_df["Target available"]
    <= selection_date
).all()

# Confirming the expanded loop reporduces the original permits forecasts
expanded_permits = (
    single_indicator_results_df[
        single_indicator_results_df["Predictor"] == "permits_yoy"
    ]
    .sort_values("Reference quarter")
    .reset_index(drop=True)
)

original_permits = (
    development_results_df
    .sort_values("Reference quarter")
    .reset_index(drop=True)
)

assert expanded_permits["Reference quarter"].equals(
    original_permits["Reference quarter"]
)

assert np.allclose(
    expanded_permits["Forecast"],
    original_permits["Forecast"]
) # This last check confirms that putting permits inside the larger loop did not change its forecasts.

print("All expanded-loop checks passed.")

All expanded-loop checks passed.


In [37]:
# Summarizing and comparing all candidates

# Calculating forecast accuracy for each candidate
single_indicator_summary = (
    single_indicator_results_df
    .groupby("Predictor")["Error"]
    .agg(
        Forecasts="count",
        MAE=lambda e: e.abs().mean(),
        RMSE=lambda e: np.sqrt((e ** 2).mean()),
        Bias="mean",
    )
    .reset_index()
    .sort_values("RMSE")
    .reset_index(drop=True)
)

# Adding the benchmark for comparison
benchmark_row = pd.DataFrame([{
    "Predictor": "historical_mean",
    "Forecasts": len(benchmark_errors),
    "MAE": benchmark_errors.abs().mean(),
    "RMSE": np.sqrt((benchmark_errors ** 2).mean()),
    "Bias": benchmark_errors.mean(),
}])

single_indicator_comparison = pd.concat(
    [single_indicator_summary, benchmark_row],
    ignore_index=True
).sort_values("RMSE").reset_index(drop=True)

display(single_indicator_comparison.round(4))

,Predictor,Forecasts,MAE,RMSE,Bias
0,permits_yoy,15,2.6930,3.9469,0.1887
1,foreign_orders_yoy,15,2.9342,4.0850,-0.0766
2,total_orders_yoy,15,3.0082,4.1545,-0.0963
3,historical_mean,15,3.0565,4.1862,-0.0801
4,ifo,15,3.2553,4.2817,-0.4662
5,construction_orders_yoy,15,3.5985,4.6150,-0.7713
6,electricity_ppi_yoy,15,3.9517,4.9139,-1.0140
7,spread_10_2,15,4.0986,4.9790,-1.2638
8,spread_10_1,15,4.2560,5.0921,-1.4821


In [38]:
# Single-indicator development results indicate that building permits YoY and foreign order YoY show the strongest results, thus far.
# However, this cannot be used to select the final two indicators on its own.
# A candidate can add useful information in a pair even if it is not the strongest on its own.
# Additionally, I still need to examine the six-quarter horizon and the later evaluation period.

In [39]:
# Exporting the new results outputted thus far using the same output_dir defined previously
output_dir.mkdir(parents=True, exist_ok=True)

benchmark_results_df.to_csv(
    output_dir / "mean_8q_development_detail.csv",
    index=False,
    date_format="%Y-%m-%d"
)

single_indicator_results_df.to_csv(
    output_dir / "single_indicators_8q_development_detail.csv",
    index=False,
    date_format="%Y-%m-%d"
)

single_indicator_comparison.to_csv(
    output_dir / "single_indicators_8q_development_comparison.csv",
    index=False
)

print("Saved benchmark and single-indicator results to:", output_dir)

Saved benchmark and single-indicator results to: C:\Users\zakar\Jupyter_Projects\4501_leading_indicators_backtests\backtest_first_run_results


In [40]:
# Adding a GDP-momentum benchmark
# The notebook already calculates gdp_momentum as trailing four-quarter GDP growth. 
# This benchmark fits a regression of future eight-quarter GDP growth on that historical momentum.
momentum_results = []

for origin in development_origins:

    forecast_date = forecast_dates.loc[origin]
    
    # Same eligible training rows as the other models
    training_filter = (
        common_complete
        & target.notna()
        & (data.index < origin)
        & (outcome_available <= forecast_date)
    )

    training_dates = data.index[training_filter]

    if len(training_dates) < minimum_training_observations:
        raise ValueError(
            f"Only {len(training_dates)} training observations "
            f"for {origin.date()}."
        )

    if pd.isna(gdp_momentum.loc[origin]):
        raise ValueError(
            f"GDP momentum is unavailable for {origin.date()}."
        )

    x_train = gdp_momentum.loc[training_dates].to_numpy()
    y_train = target.loc[training_dates].to_numpy()

    X_train = np.column_stack([
        np.ones(len(x_train)),
        x_train,
    ])

    coefficients, residuals, rank, singular_values = np.linalg.lstsq(
        X_train,
        y_train,
        rcond=None
    )

    if rank != 2:
        raise ValueError(
            f"Cannot estimate GDP-momentum model for {origin.date()}."
        )

    intercept = coefficients[0]
    momentum_coefficient = coefficients[1]

    momentum_value = gdp_momentum.loc[origin]

    prediction = (
        intercept + momentum_coefficient * momentum_value
    )

    actual = target.loc[origin]

    momentum_results.append({
        "Reference quarter": origin,
        "Forecast date": forecast_date,
        "Target quarter": target_dates.loc[
            origin, f"{horizon}q"
        ],
        "Target available": outcome_available.loc[origin],
        "Training observations": len(training_dates),
        "First training quarter": training_dates.min(),
        "Last training quarter": training_dates.max(),
        "Latest training outcome available":
            outcome_available.loc[training_dates].max(),
        "Intercept": intercept,
        "Momentum coefficient": momentum_coefficient,
        "GDP momentum": momentum_value,
        "Forecast": prediction,
        "Actual": actual,
        "Error": prediction - actual,
    })

momentum_results_df = pd.DataFrame(momentum_results)

display(momentum_results_df.head().round(4))

# This model is fitted again at each forecast date, just like the indicator regressions.
# The GDP value at the reference quarter is assumed available under the existing timing convention.

,Reference quarter,Forecast date,Target quarter,Target available,Training observations,First training quarter,Last training quarter,Latest training outcome available,Intercept,Momentum coefficient,GDP momentum,Forecast,Actual,Error
0,2015-01-01,2015-06-04,2017-01-01,2017-05-25,29,2006-01-01,2013-01-01,2015-05-25,2.9464,-0.6194,1.1612,2.2271,5.0402,-2.8131
1,2015-04-01,2015-09-03,2017-04-01,2017-08-24,30,2006-01-01,2013-04-01,2015-08-24,2.9803,-0.6218,1.7301,1.9045,5.1747,-3.2703
2,2015-07-01,2015-12-04,2017-07-01,2017-11-24,31,2006-01-01,2013-07-01,2015-11-24,3.0200,-0.6238,1.7641,1.9195,5.5412,-3.6217
3,2015-10-01,2016-03-05,2017-10-01,2018-02-24,32,2006-01-01,2013-10-01,2016-02-24,3.0756,-0.6242,1.4475,2.1720,6.0870,-3.9151
4,2016-01-01,2016-06-04,2018-01-01,2018-05-25,33,2006-01-01,2014-01-01,2016-05-25,3.1344,-0.6126,2.5357,1.5810,4.6509,-3.0699


In [41]:
# Checking and summarizing the GDP benchmark
# Confirming the benchmark uses the same origins and training counts
assert momentum_results_df["Reference quarter"].equals(
    development_results_df["Reference quarter"]
)

assert np.array_equal(
    momentum_results_df["Training observations"],
    development_results_df["Training observations"]
)

assert np.allclose(
    momentum_results_df["Actual"],
    development_results_df["Actual"]
)

assert (
    momentum_results_df["Latest training outcome available"]
    <= momentum_results_df["Forecast date"]
).all()

assert (
    momentum_results_df["Target available"]
    <= selection_date
).all()

momentum_errors = momentum_results_df["Error"]

momentum_summary = pd.DataFrame([{
    "Predictor": "gdp_momentum",
    "Forecasts": len(momentum_errors),
    "MAE": momentum_errors.abs().mean(),
    "RMSE": np.sqrt((momentum_errors ** 2).mean()),
    "Bias": momentum_errors.mean(),
}])

display(momentum_summary.round(4))

,Predictor,Forecasts,MAE,RMSE,Bias
0,gdp_momentum,15,3.2283,4.23,-0.4989


In [42]:
# The permits model's RMSE of 3.9469 is lower than both this benchmark's 4.2300 and the historical mean's 4.1862 in this development sample.
# Creating the candidate pairs
# There are eight predictor variants with 28 possible pairs; I'll exclude the combination of the two yield spreads, as they are alternative versions of the same candidate family.
from itertools import combinations

# Creating all two-indicator combinations
candidate_pairs = list(combinations(candidate_predictors, 2))

# Excluding the pair containing both yield-spread versions
candidate_pairs = [
    pair for pair in candidate_pairs
    if set(pair) != {"spread_10_2", "spread_10_1"}
]

print("Number of candidate pairs:", len(candidate_pairs))
print(candidate_pairs)

# I expect 27 candidate pairs, excluding that of the two yields together
# For example, for YoY building permits and ifo business expectations, that pair will estimate an equation of the form:
## GDP growth = intercept + b1(permits YoY) + b2(ifo expectations)
# Each coefficient now describes an association holding the other predictor in the model constant - i.e., a fitted multiple-regression model

Number of candidate pairs: 27
[('permits_yoy', 'total_orders_yoy'), ('permits_yoy', 'foreign_orders_yoy'), ('permits_yoy', 'construction_orders_yoy'), ('permits_yoy', 'electricity_ppi_yoy'), ('permits_yoy', 'ifo'), ('permits_yoy', 'spread_10_2'), ('permits_yoy', 'spread_10_1'), ('total_orders_yoy', 'foreign_orders_yoy'), ('total_orders_yoy', 'construction_orders_yoy'), ('total_orders_yoy', 'electricity_ppi_yoy'), ('total_orders_yoy', 'ifo'), ('total_orders_yoy', 'spread_10_2'), ('total_orders_yoy', 'spread_10_1'), ('foreign_orders_yoy', 'construction_orders_yoy'), ('foreign_orders_yoy', 'electricity_ppi_yoy'), ('foreign_orders_yoy', 'ifo'), ('foreign_orders_yoy', 'spread_10_2'), ('foreign_orders_yoy', 'spread_10_1'), ('construction_orders_yoy', 'electricity_ppi_yoy'), ('construction_orders_yoy', 'ifo'), ('construction_orders_yoy', 'spread_10_2'), ('construction_orders_yoy', 'spread_10_1'), ('electricity_ppi_yoy', 'ifo'), ('electricity_ppi_yoy', 'spread_10_2'), ('electricity_ppi_yoy', '

In [43]:
# Running the pair back-tests
# One two-indicator regression per pair and reference quarter
pair_results = []

for origin in development_origins:

    forecast_date = forecast_dates.loc[origin]

    # Preserving the same common training sample
    training_filter = (
        common_complete
        & target.notna()
        & (data.index < origin)
        & (outcome_available <= forecast_date)
    )

    training_dates = data.index[training_filter]

    if len(training_dates) < minimum_training_observations:
        raise ValueError(
            f"Only {len(training_dates)} training observations "
            f"for {origin.date()}."
        )

    if not common_complete.loc[origin]:
        raise ValueError(
            f"Candidate inputs are incomplete for {origin.date()}."
        )

    if pd.isna(target.loc[origin]):
        raise ValueError(
            f"GDP outcome is missing for {origin.date()}."
        )

    if outcome_available.loc[origin] > selection_date:
        raise ValueError(
            f"Outcome for {origin.date()} was unavailable "
            f"by the selection date."
        )

    y_train = target.loc[training_dates].to_numpy()

    for predictor_1, predictor_2 in candidate_pairs:

        # Two columns of candidate values for the training quarters
        x_train = features.loc[
            training_dates,
            [predictor_1, predictor_2]
        ].to_numpy()

        # Adding the intercept column
        X_train = np.column_stack([
            np.ones(len(training_dates)),
            x_train,
        ])

        coefficients, residuals, rank, singular_values = np.linalg.lstsq(
            X_train,
            y_train,
            rcond=None
        )

        # Three independent columns are required:
        # intercept + predictor 1 + predictor 2
        if rank != 3:
            raise ValueError(
                f"Cannot estimate {predictor_1} + {predictor_2} "
                f"for {origin.date()}."
            )

        intercept = coefficients[0]
        coefficient_1 = coefficients[1]
        coefficient_2 = coefficients[2]

        value_1 = features.loc[origin, predictor_1]
        value_2 = features.loc[origin, predictor_2]

        prediction = (
            intercept
            + coefficient_1 * value_1
            + coefficient_2 * value_2
        )

        actual = target.loc[origin]

        pair_results.append({
            "Predictor 1": predictor_1,
            "Predictor 2": predictor_2,
            "Pair": f"{predictor_1} + {predictor_2}",
            "Reference quarter": origin,
            "Forecast date": forecast_date,
            "Target quarter": target_dates.loc[
                origin, f"{horizon}q"
            ],
            "Target available": outcome_available.loc[origin],
            "Training observations": len(training_dates),
            "First training quarter": training_dates.min(),
            "Last training quarter": training_dates.max(),
            "Latest training outcome available":
                outcome_available.loc[training_dates].max(),
            "Intercept": intercept,
            "Coefficient 1": coefficient_1,
            "Coefficient 2": coefficient_2,
            "Predictor 1 value": value_1,
            "Predictor 2 value": value_2,
            "Forecast": prediction,
            "Actual": actual,
            "Error": prediction - actual,
        })

pair_results_df = pd.DataFrame(pair_results)

print("Pair forecast rows:", len(pair_results_df))
display(pair_results_df.head().round(4))

Pair forecast rows: 405


,Predictor 1,Predictor 2,Pair,Reference quarter,Forecast date,Target quarter,Target available,Training observations,First training quarter,Last training quarter,Latest training outcome available,Intercept,Coefficient 1,Coefficient 2,Predictor 1 value,Predictor 2 value,Forecast,Actual,Error
0,permits_yoy,total_orders_yoy,permits_yoy + total_orders_yoy,2015-01-01,2015-06-04,2017-01-01,2017-05-25,29,2006-01-01,2013-01-01,2015-05-25,2.4501,0.1352,-0.1141,-1.9178,0.6038,2.1218,5.0402,-2.9183
1,permits_yoy,foreign_orders_yoy,permits_yoy + foreign_orders_yoy,2015-01-01,2015-06-04,2017-01-01,2017-05-25,29,2006-01-01,2013-01-01,2015-05-25,2.5040,0.1350,-0.0947,-1.9178,0.3926,2.2080,5.0402,-2.8322
2,permits_yoy,construction_orders_yoy,permits_yoy + construction_orders_yoy,2015-01-01,2015-06-04,2017-01-01,2017-05-25,29,2006-01-01,2013-01-01,2015-05-25,2.3646,0.1283,-0.2113,-1.9178,-1.0139,2.3328,5.0402,-2.7074
3,permits_yoy,electricity_ppi_yoy,permits_yoy + electricity_ppi_yoy,2015-01-01,2015-06-04,2017-01-01,2017-05-25,29,2006-01-01,2013-01-01,2015-05-25,1.7145,0.1103,0.0935,-1.9178,-0.2239,1.4820,5.0402,-3.5582
4,permits_yoy,ifo,permits_yoy + ifo,2015-01-01,2015-06-04,2017-01-01,2017-05-25,29,2006-01-01,2013-01-01,2015-05-25,25.9473,0.1296,-0.2444,-1.9178,99.3333,1.4211,5.0402,-3.6190


In [44]:
# The above is 27 pairs x 15 reference quarters. 
# At each refernce quarter, every pair is fitted using the same eligible training dates.

# Validating the pair forecasts
# One forecast per pair and development origin
assert not pair_results_df.duplicated(
    ["Predictor 1", "Predictor 2", "Reference quarter"]
).any()

pair_counts = pair_results_df.groupby(
    ["Predictor 1", "Predictor 2"]
)["Reference quarter"].nunique()

assert len(pair_counts) == len(candidate_pairs)
assert (pair_counts == len(development_origins)).all()

# Training outcomes must be available at each forecast date
assert (
    pair_results_df["Latest training outcome available"]
    <= pair_results_df["Forecast date"]
).all()

# Development outcomes must be available by model-selection date
assert (
    pair_results_df["Target available"]
    <= selection_date
).all()

# Verifying the saved error calculation
assert np.allclose(
    pair_results_df["Error"],
    pair_results_df["Forecast"] - pair_results_df["Actual"]
)

# Confirming that every pair uses the same training count at an origin
expected_counts = (
    development_results_df
    .set_index("Reference quarter")["Training observations"]
)

expected_for_pairs = pair_results_df[
    "Reference quarter"
].map(expected_counts)

assert np.array_equal(
    pair_results_df["Training observations"].to_numpy(),
    expected_for_pairs.to_numpy()
)

print("All pair back-test checks passed.")

# These checks should confirm the saved structure and timing under the assumed release schedule.
# A rank of 3 confirms the coefficients are estimable (re: 3 linearly independent columns/rows), although it does not rule out strong correlation between the two indicators themselves.

All pair back-test checks passed.


In [45]:
# Ranking pairs and comparing them with benchmarks
pair_summary = (
    pair_results_df
    .groupby(["Predictor 1", "Predictor 2", "Pair"])["Error"]
    .agg(
        Forecasts="count",
        MAE=lambda e: e.abs().mean(),
        RMSE=lambda e: np.sqrt((e ** 2).mean()),
        Bias="mean",
    )
    .reset_index()
    .sort_values(["RMSE", "MAE", "Pair"])
    .reset_index(drop=True)
)

# Comparing each pair's RMSE against the historical mean
mean_rmse = benchmark_summary.loc[0, "RMSE"]

pair_summary["RMSE reduction vs mean (%)"] = (
    100 * (1 - pair_summary["RMSE"] / mean_rmse)
)

display(pair_summary.head(10).round(4))

# A positive value in RMSE reduction vs mean (%) means the pair's RMSE is lower than the historical-mean benchmark.
# A negative value means it is higher.

,Predictor 1,Predictor 2,Pair,Forecasts,MAE,RMSE,Bias,RMSE reduction vs mean (%)
0,permits_yoy,foreign_orders_yoy,permits_yoy + foreign_orders_yoy,15,2.5071,3.8493,0.2420,8.0487
1,permits_yoy,total_orders_yoy,permits_yoy + total_orders_yoy,15,2.6220,3.9351,0.2127,5.9990
2,permits_yoy,ifo,permits_yoy + ifo,15,2.8971,4.0429,-0.3242,3.4235
3,foreign_orders_yoy,ifo,foreign_orders_yoy + ifo,15,2.9938,4.1643,0.0187,0.5239
4,total_orders_yoy,ifo,total_orders_yoy + ifo,15,2.9919,4.2377,0.1582,-1.2290
5,permits_yoy,electricity_ppi_yoy,permits_yoy + electricity_ppi_yoy,15,3.0980,4.2725,-0.0149,-2.0606
6,foreign_orders_yoy,construction_orders_yoy,foreign_orders_yoy + construction_orders_yoy,15,3.2591,4.3073,-0.3985,-2.8922
7,total_orders_yoy,construction_orders_yoy,total_orders_yoy + construction_orders_yoy,15,3.2677,4.3339,-0.3397,-3.5274
8,permits_yoy,construction_orders_yoy,permits_yoy + construction_orders_yoy,15,3.3108,4.3643,-0.7354,-4.2537
9,construction_orders_yoy,ifo,construction_orders_yoy + ifo,15,3.5277,4.5149,-0.7534,-7.8521


In [46]:
# The results above indicate that building permits YoY and foreign manufacturing orders YoY are the strongest candidate pair thus far. 
# The permits on their own had an RMSE of 3.9469. Thus, in this development sample, the first two pairs slightly improve on permits alone.
# Whilst permits + ifo does not on RMSE. The differences are fairly small, relative to the size of the errors, so I'd treat this ranking as preliminary.

In [47]:
# Exporting the new results
momentum_results_df.to_csv(
    output_dir / "gdp_momentum_8q_development_detail.csv",
    index=False,
    date_format="%Y-%m-%d"
)

momentum_summary.to_csv(
    output_dir / "gdp_momentum_8q_development_summary.csv",
    index=False
)

pair_results_df.to_csv(
    output_dir / "pairs_8q_development_detail.csv",
    index=False,
    date_format="%Y-%m-%d"
)

pair_summary.to_csv(
    output_dir / "pairs_8q_development_summary.csv",
    index=False
)

print("Saved GDP-momentum and pair results to:", output_dir)

Saved GDP-momentum and pair results to: C:\Users\zakar\Jupyter_Projects\4501_leading_indicators_backtests\backtest_first_run_results


In [48]:
# Next, I'll use the eight-quarter development results to identify one fixed pair, then:
## Run the same development comparison at the six-quarter horizon.
## Evaluate that fixed pair at both horizons on later forecast origins.
## Compare it with its individual indicators and both benchmarks. 
# Established plan of using eight-quarter development RMSE as the primary selection criterion still stands.
# As such, the fixed pair is permits + foreign manufacturing orders.
# However, the six-quarter results should provide additional evidence; I (most likely) won't pick a different pair after seeing the later evaluation results.

In [49]:
# Recording selection of the pair with the lowest eight-quarter development RMSE
selected_pair_row = pair_summary.iloc[0]

selected_pair = (
    selected_pair_row["Predictor 1"],
    selected_pair_row["Predictor 2"],
)

print("Selected pair:", selected_pair)
print("Eight-quarter development RMSE:",
      round(selected_pair_row["RMSE"], 4))

# For this exercise, I'll treat the selected pair as fixed when evaluating later origins.
# Coefficients will still be re-estimated at each origin as new GDP outcomes become available.

Selected pair: ('permits_yoy', 'foreign_orders_yoy')
Eight-quarter development RMSE: 3.8493


In [50]:
# Creating a reusable back-test function.
# Currently, the notebook has separate loops for the mean, momentum, individual indicators, and pairs.
# The following should combine the same calculations such that the horizon and evaluation period may be changed without copying four separate loops each time.
def run_backtest(horizon, origins, models, evaluation_cutoff):
    """
    Expanding-window back-test for specified models and origins.

    models maps a model name to its predictor names.
    An empty tuple represents the historical-mean benchmark.
    """

    target = targets[f"gdp_growth_{horizon}q"]
    available = target_available[f"{horizon}q"]

    # GDP momentum is added to the candidate indicator columns
    model_inputs = features.copy()
    model_inputs["gdp_momentum"] = gdp_momentum

    records = []

    for origin in origins:

        forecast_date = forecast_dates.loc[origin]

        # Check that this origin can be evaluated
        if not common_complete.loc[origin]:
            raise ValueError(
                f"Inputs are incomplete for {origin.date()}."
            )

        if pd.isna(target.loc[origin]):
            raise ValueError(
                f"GDP outcome is missing for {origin.date()}."
            )

        if available.loc[origin] > evaluation_cutoff:
            raise ValueError(
                f"Outcome for {origin.date()} is unavailable "
                f"by {evaluation_cutoff.date()}."
            )

        # Select only training outcomes available at forecast time
        training_filter = (
            common_complete
            & target.notna()
            & (data.index < origin)
            & (available <= forecast_date)
        )

        training_dates = data.index[training_filter]

        if len(training_dates) < minimum_training_observations:
            raise ValueError(
                f"Only {len(training_dates)} training rows "
                f"for {origin.date()}."
            )

        y_train = target.loc[training_dates].to_numpy()

        for model_name, predictors in models.items():

            predictors = list(predictors)

            # An empty predictor list gives an intercept-only model:
            # its forecast is the historical mean.
            if len(predictors) == 0:
                prediction = float(y_train.mean())
                intercept = prediction
                coefficients = []

            else:
                x_train = model_inputs.loc[
                    training_dates, predictors
                ].to_numpy()

                X_train = np.column_stack([
                    np.ones(len(training_dates)),
                    x_train,
                ])

                fitted, residuals, rank, singular_values = np.linalg.lstsq(
                    X_train,
                    y_train,
                    rcond=None
                )

                if rank != X_train.shape[1]:
                    raise ValueError(
                        f"Cannot estimate {model_name} "
                        f"for {origin.date()}."
                    )

                current_values = model_inputs.loc[
                    origin, predictors
                ].to_numpy(dtype=float)

                intercept = fitted[0]
                coefficients = fitted[1:]

                prediction = float(
                    intercept + current_values @ coefficients
                )

            actual = float(target.loc[origin])

            record = {
                "Horizon": horizon,
                "Model": model_name,
                "Reference quarter": origin,
                "Forecast date": forecast_date,
                "Target quarter": target_dates.loc[
                    origin, f"{horizon}q"
                ],
                "Target available": available.loc[origin],
                "Training observations": len(training_dates),
                "First training quarter": training_dates.min(),
                "Last training quarter": training_dates.max(),
                "Latest training outcome available":
                    available.loc[training_dates].max(),
                "Intercept": intercept,
                "Forecast": prediction,
                "Actual": actual,
                "Error": prediction - actual,
            }

            # Preserve predictor names, values and coefficients
            for number, predictor_name in enumerate(predictors, start=1):
                record[f"Predictor {number}"] = predictor_name
                record[f"Predictor {number} value"] = (
                    model_inputs.loc[origin, predictor_name]
                )
                record[f"Coefficient {number}"] = coefficients[number - 1]

            records.append(record)

    result = pd.DataFrame(records)

    if result.empty:
        raise ValueError("No forecasts were produced.")

    return result

# The empty tuple used for the historical mean means "no predictors." 
# In an intercept-only OLS model, the fitted intercept is the mean of the training outcomes.

In [51]:
# Creating a summary function
def summarize_backtest(results):

    summary = (
        results
        .groupby(["Horizon", "Model"])["Error"]
        .agg(
            Forecasts="count",
            MAE=lambda e: e.abs().mean(),
            RMSE=lambda e: np.sqrt((e ** 2).mean()),
            Bias="mean",
        )
        .reset_index()
        .sort_values(["Horizon", "RMSE", "MAE", "Model"])
        .reset_index(drop=True)
    )

    return summary

# This should produce the previous calculated error measures

In [52]:
# Reproducing the eight-quarter result first
# Models to use for reproducibility check
check_models = {
    "historical_mean": (),
    "gdp_momentum": ("gdp_momentum",),
    "permits_yoy": ("permits_yoy",),
    "selected_pair": selected_pair,
}

check_8q = run_backtest(
    horizon=8,
    origins=development_origins,
    models=check_models,
    evaluation_cutoff=selection_date,
)

check_8q_summary = summarize_backtest(check_8q)

display(check_8q_summary.round(4))

# Compare the new function with the existing pair results
new_pair_forecasts = (
    check_8q[check_8q["Model"] == "selected_pair"]
    .sort_values("Reference quarter")
    .reset_index(drop=True)
)

old_pair_forecasts = (
    pair_results_df[
        (pair_results_df["Predictor 1"] == selected_pair[0])
        & (pair_results_df["Predictor 2"] == selected_pair[1])
    ]
    .sort_values("Reference quarter")
    .reset_index(drop=True)
)

assert new_pair_forecasts["Reference quarter"].equals(
    old_pair_forecasts["Reference quarter"]
)

assert np.allclose(
    new_pair_forecasts["Forecast"],
    old_pair_forecasts["Forecast"]
)

assert np.allclose(
    new_pair_forecasts["Actual"],
    old_pair_forecasts["Actual"]
)

print("The reusable function reproduces the existing pair forecasts.")

,Horizon,Model,Forecasts,MAE,RMSE,Bias
0,8,selected_pair,15,2.5071,3.8493,0.2420
1,8,permits_yoy,15,2.6930,3.9469,0.1887
2,8,historical_mean,15,3.0565,4.1862,-0.0801
3,8,gdp_momentum,15,3.2283,4.2300,-0.4989


The reusable function reproduces the existing pair forecasts.


In [53]:
# Running the six-quarter development comparison
# For the six-quarter analysis, I'll compare the already-selected pair with all single indicators and both benchmarks.

# Benchmarks
development_models = {
    "historical_mean": (),
    "gdp_momentum": ("gdp_momentum",),
}

# Each candidate on its own
for candidate in candidate_predictors:
    development_models[candidate] = (candidate,)

# The pair selected from eight-quarter development results
development_models["selected_pair"] = selected_pair

development_6q = run_backtest(
    horizon=6,
    origins=development_origins,
    models=development_models,
    evaluation_cutoff=selection_date,
)

development_6q_summary = summarize_backtest(development_6q)

display(development_6q_summary.round(4))

# The six-quarter target is cumulative GDP growth from each reference quarter to six quarters later.
# It is a separate regression and target from the eight-quarter model.
# The table produced may be used to inquire as to:
## Whether the selected pair outperforms the historical mean and GDP momentum.
## Whether adding foreign orders improves on permits alone.
## Whether the RMSE and MAE conclusions are correct.
# Whilst I am using the same 15 reference quarters for the development comparison,
# the training sample can differ by horizon as six-quarter outcomes mature sooner than eight-quarter outcomes.

,Horizon,Model,Forecasts,MAE,RMSE,Bias
0,6,selected_pair,15,1.3065,1.5572,-0.6465
1,6,permits_yoy,15,1.3941,1.6113,-0.6583
2,6,foreign_orders_yoy,15,1.5395,1.8164,-0.7807
3,6,total_orders_yoy,15,1.5522,1.8290,-0.7844
4,6,historical_mean,15,1.5812,1.8492,-0.7768
5,6,ifo,15,1.5973,1.8582,-0.7490
6,6,gdp_momentum,15,1.6706,1.9564,-1.0060
7,6,construction_orders_yoy,15,1.9001,2.3031,-1.1005
8,6,electricity_ppi_yoy,15,2.0839,2.4230,-1.2211
9,6,spread_10_2,15,2.3910,2.8369,-1.6731


In [54]:
# Choosing later origins whose outcomes are observable
# The later evaluation begins with 2021 Q1 reference quarters.
# An origin is eligible only if its actual GDP outcome is present and assumed available by my 22 November 2026 cutoff
later_start = pd.Timestamp("2021-01-01")
information_cutoff = pd.Timestamp("2026-11-22")

later_origins = {}

for h in (6, 8):

    horizon_target = targets[f"gdp_growth_{h}q"]
    horizon_available = target_available[f"{h}q"]

    eligible = (
        (data.index >= later_start)
        & common_complete
        & horizon_target.notna()
        & (horizon_available <= information_cutoff)
    )

    later_origins[h] = data.index[eligible]

    print(
        f"{h}Q horizon:",
        len(later_origins[h]), "origins;",
        "first =", later_origins[h].min().date(),
        "last =", later_origins[h].max().date()
    )

6Q horizon: 16 origins; first = 2021-01-01 last = 2024-10-01
8Q horizon: 14 origins; first = 2021-01-01 last = 2024-04-01


In [55]:
# The eight-quarter evaluation ends earlier because its GDP outcomes take longer to mature.
# Evaluating the fixed pair on later origins
# I'll compare the selected pair with its two individual indicators and both benchmarks
later_models = {
    "historical_mean": (),
    "gdp_momentum": ("gdp_momentum",),
    selected_pair[0]: (selected_pair[0],),
    selected_pair[1]: (selected_pair[1],),
    "selected_pair": selected_pair,
}

later_results = pd.concat(
    [
        run_backtest(
            horizon=h,
            origins=later_origins[h],
            models=later_models,
            evaluation_cutoff=information_cutoff,
        )
        for h in (6, 8)
    ],
    ignore_index=True,
)

later_summary = summarize_backtest(later_results)

display(later_summary.round(4))

# Here, the pair stays fixed, but the coefficients are refitted at every historical forecast date using the outcomes then available.
# This represents a forecasting process that updates its model over time

,Horizon,Model,Forecasts,MAE,RMSE,Bias
0,6,permits_yoy,16,0.9258,1.1182,0.2298
1,6,selected_pair,16,0.9548,1.1926,0.3051
2,6,historical_mean,16,1.6092,1.8209,1.2878
3,6,foreign_orders_yoy,16,1.7610,1.9798,1.4402
4,6,gdp_momentum,16,1.9359,2.0584,1.3858
5,8,permits_yoy,14,1.6652,1.7746,0.1242
6,8,selected_pair,14,1.6832,1.9742,0.0157
7,8,historical_mean,14,2.0155,2.1719,1.8899
8,8,gdp_momentum,14,2.4688,2.6838,1.8910
9,8,foreign_orders_yoy,14,2.4943,2.7042,1.9983


In [56]:
# Final checks and exporting the results
# Timing and arithmetic checks
assert (
    later_results["Latest training outcome available"]
    <= later_results["Forecast date"]
).all()

assert (
    later_results["Target available"]
    <= information_cutoff
).all()

assert np.allclose(
    later_results["Error"],
    later_results["Forecast"] - later_results["Actual"]
)

# Every model must be evaluated on the same origins within each horizon
for h in (6, 8):

    horizon_rows = later_results[
        later_results["Horizon"] == h
    ]

    origin_sets = horizon_rows.groupby("Model")[
        "Reference quarter"
    ].apply(lambda dates: tuple(sorted(dates)))

    assert all(
        origins == origin_sets.iloc[0]
        for origins in origin_sets
    )

# Export snapshots
development_6q.to_csv(
    output_dir / "development_6q_selected_models_detail.csv",
    index=False,
    date_format="%Y-%m-%d"
)

development_6q_summary.to_csv(
    output_dir / "development_6q_selected_models_summary.csv",
    index=False
)

later_results.to_csv(
    output_dir / "later_evaluation_detail.csv",
    index=False,
    date_format="%Y-%m-%d"
)

later_summary.to_csv(
    output_dir / "later_evaluation_summary.csv",
    index=False
)

print("Checks passed. Results saved to:", output_dir)

Checks passed. Results saved to: C:\Users\zakar\Jupyter_Projects\4501_leading_indicators_backtests\backtest_first_run_results


In [57]:
# Running the timing sensitivity
# The main back-test used indicator values from the GDP reference quarter.
# This sensitiviy repeats the later evaluation using indicators one quarter older, whilst keeping the same target quarters and fixed pair.
# For example, for a forecast anchored on 2021 Q1, the sensitivity uses the indicators' 2020 Q4 values.

# Timing sensitivity: candidate indicators one quarter older
lagged_features = features.shift(1)

# Keeping GDP momentum aligned with the GDP reference quarter
lagged_inputs = lagged_features.copy()
lagged_inputs["gdp_momentum"] = gdp_momentum

# Common sample for the lagged scenario
lagged_complete = lagged_inputs.notna().all(axis=1)

sensitivity_models = {
    "historical_mean": (),
    "gdp_momentum": ("gdp_momentum",),
    selected_pair[0]: (selected_pair[0],),
    selected_pair[1]: (selected_pair[1],),
    "selected_pair": selected_pair,
}

sensitivity_records = []

for h in (6, 8):

    sensitivity_target = targets[f"gdp_growth_{h}q"]
    sensitivity_available = target_available[f"{h}q"]

    for origin in later_origins[h]:

        forecast_date = forecast_dates.loc[origin]

        if not lagged_complete.loc[origin]:
            raise ValueError(
                f"Lagged inputs missing for {origin.date()}."
            )

        training_filter = (
            lagged_complete
            & sensitivity_target.notna()
            & (data.index < origin)
            & (sensitivity_available <= forecast_date)
        )

        training_dates = data.index[training_filter]

        if len(training_dates) < minimum_training_observations:
            raise ValueError(
                f"Too few training observations for {origin.date()}."
            )

        y_train = sensitivity_target.loc[
            training_dates
        ].to_numpy()

        for model_name, predictors in sensitivity_models.items():

            predictors = list(predictors)

            if len(predictors) == 0:
                prediction = float(y_train.mean())

            else:
                x_train = lagged_inputs.loc[
                    training_dates, predictors
                ].to_numpy()

                X_train = np.column_stack([
                    np.ones(len(training_dates)),
                    x_train,
                ])

                fitted, residuals, rank, singular_values = np.linalg.lstsq(
                    X_train,
                    y_train,
                    rcond=None
                )

                if rank != X_train.shape[1]:
                    raise ValueError(
                        f"Cannot estimate {model_name} "
                        f"for {origin.date()}."
                    )

                current_values = lagged_inputs.loc[
                    origin, predictors
                ].to_numpy(dtype=float)

                prediction = float(
                    fitted[0] + current_values @ fitted[1:]
                )

            actual = float(sensitivity_target.loc[origin])

            sensitivity_records.append({
                "Horizon": h,
                "Model": model_name,
                "Reference quarter": origin,
                "Forecast date": forecast_date,
                "Target quarter": target_dates.loc[origin, f"{h}q"],
                "Target available": sensitivity_available.loc[origin],
                "Training observations": len(training_dates),
                "Latest training outcome available":
                    sensitivity_available.loc[training_dates].max(),
                "Forecast": prediction,
                "Actual": actual,
                "Error": prediction - actual,
            })

sensitivity_results = pd.DataFrame(sensitivity_records)
sensitivity_summary = summarize_backtest(sensitivity_results)

# Timing and comparison checks
assert (
    sensitivity_results["Latest training outcome available"]
    <= sensitivity_results["Forecast date"]
).all()

assert (
    sensitivity_results["Target available"]
    <= information_cutoff
).all()

for h in (6, 8):
    rows = sensitivity_results[sensitivity_results["Horizon"] == h]

    origin_sets = rows.groupby("Model")[
        "Reference quarter"
    ].apply(lambda dates: tuple(sorted(dates)))

    assert all(x == origin_sets.iloc[0] for x in origin_sets)

display(sensitivity_summary.round(4))

# The lagged scenario uses a common sample across all its models.
# Its eligible training rows may differ slightly from the original scenario, as shifting the indicators changes which rows are complete.

,Horizon,Model,Forecasts,MAE,RMSE,Bias
0,6,permits_yoy,16,1.3171,1.4528,0.2085
1,6,selected_pair,16,1.4026,1.7352,0.0229
2,6,historical_mean,16,1.5626,1.7811,1.2283
3,6,gdp_momentum,16,1.8979,2.0307,1.3139
4,6,foreign_orders_yoy,16,1.9708,2.1882,1.1799
5,8,historical_mean,14,1.9509,2.1060,1.8130
6,8,permits_yoy,14,1.8911,2.1760,0.3307
7,8,gdp_momentum,14,2.4131,2.6537,1.7932
8,8,selected_pair,14,2.0219,2.7991,-0.0838
9,8,foreign_orders_yoy,14,2.6300,2.9899,1.6472


In [58]:
# Refitting the final models as of 22 November 2026
## GDP anchor: 2026 Q2
## Predictors: 2026 Q2 permits YoY and foreign manufacturing orders YoY
## Training outcomes: only those assumed available by 22 November 2026
## Separate OLS equations for six and eight quarters
# I've decided to omit Q3 observations from this final fit and forecast, as several of the values are partial, and the real GDP index dataset has not yet been updated with Q3 2026 values.

# Final forecast settings
final_cutoff = pd.Timestamp("2026-11-22")
final_anchor = pd.Timestamp("2026-04-01")  # 2026 Q2

final_predictors = list(selected_pair)

# Guard against accidentally changing the chosen pair
assert final_predictors == [
    "permits_yoy",
    "foreign_orders_yoy",
]

anchor_gdp = data.loc[final_anchor, "gdp"]
anchor_values = features.loc[
    final_anchor, final_predictors
].to_numpy(dtype=float)

if pd.isna(anchor_gdp) or np.isnan(anchor_values).any():
    raise ValueError("The GDP anchor or predictor values are missing.")

final_forecast_records = []
final_training_records = []

for h in (6, 8):

    final_target = targets[f"gdp_growth_{h}q"]
    final_available = target_available[f"{h}q"]

    # Same common-sample convention used in the main back-test
    training_filter = (
        common_complete
        & final_target.notna()
        & (data.index < final_anchor)
        & (final_available <= final_cutoff)
    )

    training_dates = data.index[training_filter]

    if len(training_dates) < minimum_training_observations:
        raise ValueError(f"Too few training observations for {h}Q.")

    x_train = features.loc[
        training_dates, final_predictors
    ].to_numpy()

    y_train = final_target.loc[training_dates].to_numpy()

    X_train = np.column_stack([
        np.ones(len(training_dates)),
        x_train,
    ])

    coefficients, residuals, rank, singular_values = np.linalg.lstsq(
        X_train,
        y_train,
        rcond=None
    )

    if rank != 3:
        raise ValueError(f"Cannot estimate the final {h}Q model.")

    predicted_growth = float(
        coefficients[0] + anchor_values @ coefficients[1:]
    )

    implied_gdp_index = float(
        anchor_gdp * (1 + predicted_growth / 100)
    )

    fitted_training = X_train @ coefficients

    for i, training_origin in enumerate(training_dates):
        final_training_records.append({
            "Horizon": h,
            "Reference quarter": training_origin,
            "Target quarter": target_dates.loc[
                training_origin, f"{h}q"
            ],
            "Target available": final_available.loc[training_origin],
            "Actual cumulative growth": y_train[i],
            "Fitted cumulative growth": fitted_training[i],
            "Residual": fitted_training[i] - y_train[i],
        })

    final_forecast_records.append({
        "Cutoff date": final_cutoff,
        "GDP anchor quarter": final_anchor,
        "GDP anchor index": anchor_gdp,
        "Horizon": h,
        "Target quarter": target_dates.loc[final_anchor, f"{h}q"],
        "Training observations": len(training_dates),
        "Last training reference quarter": training_dates.max(),
        "Latest training outcome available":
            final_available.loc[training_dates].max(),
        "Intercept": coefficients[0],
        "Permits YoY": anchor_values[0],
        "Permits coefficient": coefficients[1],
        "Foreign orders YoY": anchor_values[1],
        "Foreign orders coefficient": coefficients[2],
        "Predicted cumulative GDP growth (%)": predicted_growth,
        "Implied GDP index": implied_gdp_index,
    })

final_forecasts = pd.DataFrame(final_forecast_records)
final_training_fit = pd.DataFrame(final_training_records)

# Final timing check
assert (
    final_forecasts["Latest training outcome available"]
    <= final_forecasts["Cutoff date"]
).all()

display(final_forecasts.round(4))

,Cutoff date,GDP anchor quarter,GDP anchor index,Horizon,Target quarter,Training observations,Last training reference quarter,Latest training outcome available,Intercept,Permits YoY,Permits coefficient,Foreign orders YoY,Foreign orders coefficient,Predicted cumulative GDP growth (%),Implied GDP index
0,2026-11-22,2026-04-01,106.26,6,2027-10-01,76,2024-10-01,2026-08-24,1.8196,9.8755,0.0744,3.7472,-0.0311,2.4379,108.8505
1,2026-11-22,2026-04-01,106.26,8,2028-04-01,74,2024-04-01,2026-08-24,2.5056,9.8755,0.1146,3.7472,-0.0821,3.3294,109.7978


In [59]:
# Exporting results and potential-chart data
# This cell exports:
## Sensitivity results
## Final coefficients and point forecasts
## In-sample fitted values for illustrating each fitted model
## Historical quarterly data for an Excel baseline chart
## Separate forecast points for a chart

# Historical chart data, with values dated to their actual quarters
chart_history = pd.DataFrame({
    "Quarter": data.index,
    "Real GDP index": data["gdp"].to_numpy(),
    "Real GDP trailing 4Q growth (%)": gdp_momentum.to_numpy(),
    "Building permits YoY (%)": features["permits_yoy"].to_numpy(),
    "Foreign manufacturing orders YoY (%)":
        features["foreign_orders_yoy"].to_numpy(),
})

# Omitted data after the 2026 Q2 anchor for this project's convention
chart_history = chart_history[
    chart_history["Quarter"] <= final_anchor
].copy()

# Separating forecast points: these are not quarterly forecasts
chart_forecast_points = final_forecasts[[
    "GDP anchor quarter",
    "GDP anchor index",
    "Horizon",
    "Target quarter",
    "Predicted cumulative GDP growth (%)",
    "Implied GDP index",
]].copy()

sensitivity_results.to_csv(
    output_dir / "timing_sensitivity_detail.csv",
    index=False,
    date_format="%Y-%m-%d"
)

sensitivity_summary.to_csv(
    output_dir / "timing_sensitivity_summary.csv",
    index=False
)

final_forecasts.to_csv(
    output_dir / "final_november_forecasts.csv",
    index=False,
    date_format="%Y-%m-%d"
)

final_training_fit.to_csv(
    output_dir / "final_models_training_fitted_values.csv",
    index=False,
    date_format="%Y-%m-%d"
)

chart_history.to_csv(
    output_dir / "excel_chart_history.csv",
    index=False,
    date_format="%Y-%m-%d"
)

chart_forecast_points.to_csv(
    output_dir / "excel_chart_forecast_points.csv",
    index=False,
    date_format="%Y-%m-%d"
)

print("Saved results and chart data to:", output_dir)

Saved results and chart data to: C:\Users\zakar\Jupyter_Projects\4501_leading_indicators_backtests\backtest_first_run_results
